# Day 051 — Exercise 3: Wiring State to the Model

**What you'll build:** `build_messages(state, user_text)` and `chat_with_history(state, user_text, model)` — the functions the app calls when the user submits a message.

**Why it matters:** A chat app must send the *whole* conversation to the model each turn — the system prompt, the prior turns, and the new one — or the model forgets everything. `build_messages` assembles that list without mutating state, and `chat_with_history` calls Ollama with a try/except fallback so a model hiccup never takes the UI down.

## Provided: Setup + Session State + Input Handling

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import ollama


def init_session(state: dict) -> dict:
    """
    Idempotently initialise a Streamlit-style session_state dict.

    Streamlit reruns the WHOLE script top-to-bottom on every interaction, so
    initialisation must never overwrite existing data. Only set a key if absent.

    Ensures keys:
        'messages'  -> list of {'role', 'content'} dicts (starts empty)
        'settings'  -> {'model', 'temperature', 'system_prompt'}
    Returns the same dict, mutated in place.
    """
    if 'messages' not in state:
        state['messages'] = []
    if 'settings' not in state:
        state['settings'] = {
            'model': 'llama3.2',
            'temperature': 0.7,
            'system_prompt': 'You are a helpful assistant.',
        }
    return state


def add_message(state: dict, role: str, content: str) -> dict:
    """Append a {'role', 'content'} message to state['messages']; return it."""
    if role not in ('user', 'assistant', 'system'):
        raise ValueError(f'invalid role: {role!r}')
    msg = {'role': role, 'content': content}
    state['messages'].append(msg)
    return msg


def reset_messages(state: dict) -> None:
    """Clear the conversation but keep settings (a 'Clear chat' button)."""
    state['messages'] = []


def validate_user_input(text: str, max_chars: int = 2000) -> tuple[bool, str]:
    """
    Validate raw text from an st.chat_input / st.text_area widget before it is
    sent to the model.

    Returns (is_valid, result):
      - empty/whitespace : (False, 'Please enter a message.')
      - too long         : (False, 'Message too long (max N chars).')
      - valid            : (True, cleaned_text)   # stripped
    """
    cleaned = text.strip()
    if not cleaned:
        return (False, 'Please enter a message.')
    if len(cleaned) > max_chars:
        return (False, f'Message too long (max {max_chars} chars).')
    return (True, cleaned)


def clamp(value: float, lo: float, hi: float) -> float:
    """Clamp a widget value into [lo, hi]. st.slider bounds live input, but a
    value restored from session_state or a URL param may be out of range."""
    return max(lo, min(hi, value))


def build_settings(model: str, temperature: float, system_prompt: str) -> dict:
    """
    Assemble a validated settings dict from sidebar widget values.
    - temperature clamped to [0.0, 1.0]
    - system_prompt stripped; empty falls back to a default
    """
    sp = system_prompt.strip() or 'You are a helpful assistant.'
    return {
        'model': model,
        'temperature': float(clamp(temperature, 0.0, 1.0)),
        'system_prompt': sp,
    }

## Your Implementation

In [ ]:
def build_messages(state: dict, user_text: str) -> list:
    """
    Build the ollama.chat messages list:
        [system_prompt] + prior conversation + new user turn.
    Read the system prompt from state['settings']. Do NOT mutate state.
    """
    settings = state.get('settings', {})
    system_prompt = settings.get('system_prompt', 'You are a helpful assistant.')
    # TODO: messages = [{'role': 'system', 'content': system_prompt}]
    # TODO: messages.extend(state.get('messages', []))
    # TODO: messages.append({'role': 'user', 'content': user_text})
    # TODO: return messages
    pass


def chat_with_history(state: dict, user_text: str, model: str = 'llama3.2') -> str:
    """
    Send the conversation to Ollama; return the assistant reply.
    Read temperature from state['settings']. Return a fallback string on error.
    """
    settings = state.get('settings', {})
    temperature = settings.get('temperature', 0.7)
    messages = build_messages(state, user_text)
    # TODO: try: call ollama.chat(model=model, messages=messages,
    #                              options={'temperature': temperature})
    #        return response['message']['content'].strip()
    # TODO: except Exception as e: return f'[Model unavailable: {e}]'
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    state = init_session({})
    add_message(state, 'user', 'first question')
    add_message(state, 'assistant', 'first answer')

    # Check 1: build_messages returns a list starting with system, ending with user
    try:
        msgs = build_messages(state, 'second question')
        assert isinstance(msgs, list), f'expected list, got {type(msgs).__name__}'
        assert msgs[0]['role'] == 'system', 'first message must be system'
        assert msgs[-1] == {'role': 'user', 'content': 'second question'}, 'last must be new user turn'
        passed += 1; print('✅ Check 1: build_messages system-first, user-last')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: prior conversation is included in the middle
    try:
        msgs = build_messages(state, 'second question')
        contents = [m['content'] for m in msgs]
        assert 'first question' in contents and 'first answer' in contents, 'history missing'
        assert len(msgs) == 4, f'expected system+2 history+user = 4, got {len(msgs)}'
        passed += 1; print('✅ Check 2: prior history included')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: build_messages does NOT mutate state
    try:
        before = len(state['messages'])
        build_messages(state, 'temp')
        assert len(state['messages']) == before, 'build_messages mutated state!'
        passed += 1; print('✅ Check 3: build_messages leaves state unchanged')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: chat_with_history returns a non-empty string
    try:
        reply = chat_with_history(state, 'Reply with the single word: pong')
        assert isinstance(reply, str) and len(reply) > 0, f'expected non-empty str, got {reply!r}'
        passed += 1; print(f'✅ Check 4: chat_with_history -> str ({len(reply)} chars)')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: a bad model name is handled gracefully (fallback string, no crash)
    try:
        reply = chat_with_history(state, 'hi', model='no-such-model-xyz')
        assert isinstance(reply, str), 'must return a string even on model error'
        passed += 1; print('✅ Check 5: model error handled gracefully')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def build_messages(state: dict, user_text: str) -> list:
    """
    Build the messages list for ollama.chat:
        [system_prompt] + prior conversation + new user turn.
    Reads the system prompt from state['settings']. Does NOT mutate state.
    """
    settings = state.get('settings', {})
    system_prompt = settings.get('system_prompt', 'You are a helpful assistant.')
    messages = [{'role': 'system', 'content': system_prompt}]
    messages.extend(state.get('messages', []))
    messages.append({'role': 'user', 'content': user_text})
    return messages


def chat_with_history(state: dict, user_text: str, model: str = 'llama3.2') -> str:
    """
    Send the full conversation to Ollama and return the assistant's reply.
    Reads temperature from state['settings']. Returns a fallback string if
    Ollama is unavailable so the app never crashes on a model error.
    """
    settings = state.get('settings', {})
    temperature = settings.get('temperature', 0.7)
    messages = build_messages(state, user_text)
    try:
        response = ollama.chat(
            model=model,
            messages=messages,
            options={'temperature': temperature},
        )
        return response['message']['content'].strip()
    except Exception as e:
        return f'[Model unavailable: {e}]' 
```

**Why this works:** `build_messages` reads from state but returns a fresh list — it never appends to `state['messages']`, so the app controls exactly when history is committed. `chat_with_history` wraps the network call in try/except and returns a string on *every* path, so the UI can always render a reply — a crashed model becomes a visible message, not a 500 page. `options={'temperature': ...}` is how Ollama takes sampling parameters.
</details>